# Démo de suppression de SRA Synapse
## — Extrayez physiquement des connaissances spécifiques d'un LLM !

Ce cahier est une démo interactive de l'article *[\[AI Romance\] Extraire physiquement des connaissances spécifiques d'un LLM ! « Suppression de synapse » et « purge » du LLM remplaçables à chaud](https://qiita.com/JunSuzukiJapan/items/)*.

Dans SRA (Synaptic Routing Architecture), les connaissances et synapses inutiles peuvent être supprimées de deux manières.

| Méthode | API | Objectif |
|------|-----|------|
| **Retrait physique** | `pop_synapses(N)` | Supprime les synapses ajoutées via Hot-Swap de la queue pour restaurer la taille du modèle |
| **Zéro-effacement (purge)** | `clear_synapses([idx])` | Désactive les synapses aux indices intermédiaires et les convertit en emplacements libres |

Nous confirmerons également le **piège de similarité cosinus** qui se produit lors de la compensation des zéros, et sa solution, dans la pratique.

Enfin, nous appliquons `clear_synapses` à un modèle réellement entraîné multitâche et démontrons que **seule la fonction de la tâche ciblée est détruite tandis que toutes les autres tâches restent totalement intactes (zéro oubli)**.

---
## Partie 1 : Suppression des synapses (`pop_synapses`)

Nous avons physiquement coupé les synapses qui ont été ajoutées ultérieurement via Hot-Swap, en commençant par la queue.
Tout comme la désinstallation d'un plugin d'un système d'exploitation, vous pouvez supprimer physiquement des parties du cerveau de l'IA.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## Partie 2 : Remise à zéro (purge) et « piège à similarité cosinus »

Si nous voulons supprimer un Synapse à un index intermédiaire, le supprimer physiquement déplacerait les identifiants.
Au lieu de cela, nous écrasons les poids avec `0.0` — un « zéro-effacement ».

Cependant, il y a un **piège terrifiant** ici. La similarité cosinus du vecteur zéro devient `0.0`,
et si les scores des synapses normales sont négatifs, la **synapse vide se retrouve avec un score plus élevé et commence à absorber les données** — un phénomène de trou noir.

Le routeur de SRA dispose d'un **masque `-inf`** intégré pour éviter ce piège. Vérifions-le en pratique.

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## Partie 3 : Preuve fonctionnelle — `clear_synapses` sur un modèle entraîné

Passons maintenant à l'événement principal. Dans les parties 1 et 2, nous avons vérifié le "comportement de l'API",
mais ce qui compte vraiment, c'est la preuve fonctionnelle : **"Après la compensation à zéro, seules les connaissances supprimées sont-elles perdues tandis que toutes les autres connaissances restent totalement intactes ?"**

En utilisant la même approche que le cahier 05 (l'expérience Lésion) :
1. Train multitâche sur `copy` et `reverse`
2. Identifiez les synapses fréquemment utilisées par `reverse` et supprimez-les avec `clear_synapses`
3. Vérifiez que `reverse` devient insoluble tandis que `copy` continue d'obtenir un score de 100 % (zéro oubli).

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. Test de performances avant suppression et identification de la cible
Nous confirmons que chaque tâche peut être résolue correctement et identifions les synapses les plus utilisées par `reverse`.

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. Suppression de synapse via `clear_synapses`
Dans le notebook 05, nous avons exécuté manuellement `block.w1.data[idx].zero_()`, mais ici nous utilisons l'**API `clear_synapses`** officielle, qui applique également le masque `-inf` du routeur, pour effectuer une suppression en toute sécurité.

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


### 3-3. Test de performances après suppression (vérification du zéro oubli)

Nous testons à nouveau avec certaines synapses supprimées.

**Résultats attendus :**
- **Copie** : précision préservée (utilise différentes synapses, donc complètement intactes)
- **Reverse** : la précision chute (ses synapses spécialisées ont disparu)

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## Résumé

| Démo | Ce qui a été démontré |
|------|----------------------|
| Partie 1 | Suppression physique et restauration de la mémoire via `pop_synapses` |
| Partie 2 | Remise à zéro via `clear_synapses` et vérification du masque `-inf` |
| Partie 3 | `clear_synapses` sur un modèle entraîné -> seule la tâche ciblée est détruite tandis que les autres restent intactes |

Grâce à cela, nous avons réalisé le cycle de vie complet d'une IA modulaire : **"formation -> intégration (Hot-Swap) -> suppression (purge) -> réutilisation des emplacements (écrasement)"**.